#Preparación datasets

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

# ── Semilla global ──────────────────────────────────────────────
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Semillas fijadas con SEED={SEED}")

In [ ]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

# ── Carga ───────────────────────────────────────────────────────
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

# ── Split fijo train / validación ───────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full      # mantiene proporción de clases
)

# ── Verificación ─────────────────────────────────────────────────
print(f"Train:      {X_train.shape}  |  {y_train.shape}")
print(f"Validación: {X_val.shape}    |  {y_val.shape}")
print(f"Test:       {X_test.shape}   |  {y_test.shape}")

CLASES = ['airplane','automobile','bird','cat','deer',
          'dog','frog','horse','ship','truck']
print(f"\nClases: {CLASES}")

#Normalización base (compartida por ambos baselines)

In [ ]:

X_train = X_train.astype("float32") / 255.0
X_val   = X_val.astype("float32")   / 255.0
X_test  = X_test.astype("float32")  / 255.0

print(f"Rango train: [{X_train.min():.1f}, {X_train.max():.1f}]")
print(f"Dtype: {X_train.dtype}")

# Baseline A

In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf

def build_baseline_A():
    tf.random.set_seed(SEED)

    model = models.Sequential([
        # Bloque 1
        layers.Conv2D(32, (3,3), activation='relu',
                      padding='same', input_shape=(32,32,3)),
        layers.MaxPooling2D(2,2),

        # Bloque 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),

        # Bloque 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),

        # Head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5, seed=SEED),
        layers.Dense(10, activation='softmax')

    ], name="Baseline_A")

    return model

model_A = build_baseline_A()
model_A.summary()

In [ ]:
import os
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

os.makedirs("checkpoints", exist_ok=True)

def get_callbacks(run_name):
    """
    run_name: identificador único del experimento, ej. 'A_baseline', 'B_exp1'
    """
    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,  # recupera los pesos del mejor epoch
        verbose=1
    )

    checkpoint = ModelCheckpoint(
        filepath=f"checkpoints/{run_name}_best.keras",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )

    return [early_stop, checkpoint]

print("Callbacks definidos")
print("   → EarlyStopping:    patience=5, monitor=val_accuracy")
print("   → ModelCheckpoint:  guarda checkpoints/<run_name>_best.keras")

#Entrenamineto Baseline A

In [ ]:
# ── Compilar ─────────────────────────────────────────────────────
model_A = build_baseline_A()

model_A.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # labels son enteros, no one-hot
    metrics=['accuracy']
)

# ── Entrenar ─────────────────────────────────────────────────────
history_A_baseline = model_A.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("A_baseline"),
    verbose=1
)

# ── Resultado ────────────────────────────────────────────────────
best_val_acc_A = max(history_A_baseline.history['val_accuracy'])
print(f"\n🏆 Baseline A — Mejor val_accuracy: {best_val_acc_A:.4f}")

#Baseline B: Arquitectura MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Input
from tensorflow.keras import layers, models

def build_baseline_B():
    tf.random.set_seed(SEED)

    # ── Input + resize interno ───────────────────────────────────
    inputs = Input(shape=(32, 32, 3), name="input")
    x = layers.Resizing(96, 96)(inputs)

    # ── Backbone congelado ───────────────────────────────────────
    backbone = MobileNetV2(
        input_shape=(96, 96, 3),
        include_top=False,
        weights='imagenet'
    )
    backbone.trainable = False  # solo entrenamos el head

    x = backbone(x, training=False)

    # ── Head ─────────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3, seed=SEED)(x)
    outputs = layers.Dense(10, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="Baseline_B")
    return model

model_B = build_baseline_B()
model_B.summary()

#Entrenamiento Baseline B

In [ ]:
# ── Compilar ─────────────────────────────────────────────────────
model_B = build_baseline_B()

model_B.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Entrenar ─────────────────────────────────────────────────────
history_B_baseline = model_B.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("B_baseline"),
    verbose=1
)

# ── Resultado ────────────────────────────────────────────────────
best_val_acc_B = max(history_B_baseline.history['val_accuracy'])
print(f"\n🏆 Baseline B — Mejor val_accuracy: {best_val_acc_B:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title, run_name=None):
    """Dibuja curvas accuracy y loss (train vs val) de un historial de entrenamiento."""
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs   = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13)

    # Accuracy
    ax1.plot(epochs, acc,     'b-o', markersize=3, label='Train')
    ax1.plot(epochs, val_acc, 'r-o', markersize=3, label='Validación')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(epochs, loss,     'b-o', markersize=3, label='Train')
    ax2.plot(epochs, val_loss, 'r-o', markersize=3, label='Validación')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if run_name:
        plt.savefig(f"checkpoints/{run_name}_curves.png", dpi=100)
    plt.show()

print("plot_history() definida")


# Curvas de Entrenamiento — Baselines

In [ ]:
# Curvas de entrenamiento — Baselines A y B
plot_history(history_A_baseline, "Baseline A — CNN desde cero",       "A_baseline")
plot_history(history_B_baseline, "Baseline B — MobileNetV2 (congelado)", "B_baseline")


Experimento A-1: Learning Rate

In [ ]:
# ── Experimento A-1: lr=0.0003 ───────────────────────────────────
model_A1 = build_baseline_A()

model_A1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_A1 = model_A1.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("A_exp1_lr0003"),
    verbose=1
)

best_A1 = max(history_A1.history['val_accuracy'])
print(f"\n🧪 A-1 (lr=0.0003) — val_accuracy: {best_A1:.4f}")

#Experimento A-2: Data Augmentation

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom

def build_augmentation():
    return models.Sequential([
        RandomFlip("horizontal", seed=SEED),
        RandomRotation(0.1, seed=SEED),
        RandomZoom(0.1, seed=SEED)
    ], name="augmentation")

augmentation = build_augmentation()

model_A2 = build_baseline_A()

model_A2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Construir input con augmentation aplicada solo en training
inputs = tf.keras.Input(shape=(32, 32, 3))
x = augmentation(inputs, training=True)
outputs = model_A2(x)
model_A2_aug = tf.keras.Model(inputs, outputs, name="A_exp2_augmentation")

model_A2_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_A2 = model_A2_aug.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("A_exp2_augmentation"),
    verbose=1
)

best_A2 = max(history_A2.history['val_accuracy'])
print(f"A-2 (augmentation) -- val_accuracy: {best_A2:.4f}")

#Experimento A-3: Reducir Dropout

In [ ]:
def build_baseline_A_dropout03():
    tf.random.set_seed(SEED)

    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=(32,32,3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3, seed=SEED),  # cambiado de 0.5 a 0.3
        layers.Dense(10, activation='softmax')
    ], name="A_exp3_dropout03")

    return model

model_A3 = build_baseline_A_dropout03()

model_A3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_A3 = model_A3.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("A_exp3_dropout03"),
    verbose=1
)

best_A3 = max(history_A3.history['val_accuracy'])
print(f"A-3 (dropout=0.3) -- val_accuracy: {best_A3:.4f}")

#Experimento A-4: ReduceLROnPlateau

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

def get_callbacks_with_scheduler(run_name):
    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )

    checkpoint = ModelCheckpoint(
        filepath=f"checkpoints/{run_name}_best.keras",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,       # divide el lr a la mitad
        patience=3,       # tras 3 epochs sin mejora
        min_lr=1e-6,
        verbose=1
    )

    return [early_stop, checkpoint, reduce_lr]

model_A4 = build_baseline_A()

model_A4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_A4 = model_A4.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks_with_scheduler("A_exp4_reducelr"),
    verbose=1
)

best_A4 = max(history_A4.history['val_accuracy'])
print(f"A-4 (ReduceLROnPlateau) -- val_accuracy: {best_A4:.4f}")

#Experimento B-1: Learning Rate → 0.0001

In [ ]:
model_B1 = build_baseline_B()

model_B1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_B1 = model_B1.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("B_exp1_lr00001"),
    verbose=1
)

best_B1 = max(history_B1.history['val_accuracy'])
print(f"B-1 (lr=0.0001) -- val_accuracy: {best_B1:.4f}")

#Experimento B-2: Data Augmentation

In [ ]:
augmentation_B = models.Sequential([
    layers.RandomFlip("horizontal", seed=SEED)
], name="augmentation_B")

inputs = tf.keras.Input(shape=(32, 32, 3))
x = augmentation_B(inputs, training=True)
x = build_baseline_B()(x)

model_B2 = tf.keras.Model(inputs, x, name="B_exp2_flip")

model_B2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_B2 = model_B2.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("B_exp2_flip"),
    verbose=1
)

best_B2 = max(history_B2.history['val_accuracy'])
print(f"B-2 (flip horizontal) -- val_accuracy: {best_B2:.4f}")

#Experimento B-3: Fine-tuning

In [ ]:
def build_baseline_B_finetuned():
    tf.random.set_seed(SEED)

    inputs = tf.keras.Input(shape=(32, 32, 3), name="input")
    x = layers.Resizing(96, 96)(inputs)

    backbone = MobileNetV2(
        input_shape=(96, 96, 3),
        include_top=False,
        weights='imagenet'
    )

    # Congelar todas las capas menos las ultimas 30
    backbone.trainable = True
    for layer in backbone.layers[:-30]:
        layer.trainable = False

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3, seed=SEED)(x)
    outputs = layers.Dense(10, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name="B_exp3_finetune30")
    return model

model_B3 = build_baseline_B_finetuned()

trainable = sum([tf.size(w).numpy() for w in model_B3.trainable_weights])
print(f"Parametros entrenables: {trainable:,}")

model_B3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_B3 = model_B3.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks("B_exp3_finetune30"),
    verbose=1
)

best_B3 = max(history_B3.history['val_accuracy'])
print(f"B-3 (fine-tuning top 30) -- val_accuracy: {best_B3:.4f}")

#Experimento B-4: ReduceLROnPlateau

In [ ]:
model_B4 = build_baseline_B()

model_B4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_B4 = model_B4.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=get_callbacks_with_scheduler("B_exp4_reducelr"),
    verbose=1
)

best_B4 = max(history_B4.history['val_accuracy'])
print(f"B-4 (ReduceLROnPlateau) -- val_accuracy: {best_B4:.4f}")

# Curvas de Entrenamiento — Experimentos A y B

In [ ]:
# Curvas de entrenamiento — Experimentos A
plot_history(history_A1, "A-1: lr=0.0003",              "A_exp1")
plot_history(history_A2, "A-2: Augmentation",            "A_exp2")
plot_history(history_A3, "A-3: Dropout=0.3",             "A_exp3")
plot_history(history_A4, "A-4: ReduceLROnPlateau",       "A_exp4")

# Curvas de entrenamiento — Experimentos B
plot_history(history_B1, "B-1: lr=0.0001",              "B_exp1")
plot_history(history_B2, "B-2: Flip horizontal",         "B_exp2")
plot_history(history_B3, "B-3: Fine-tuning top 30",      "B_exp3")
plot_history(history_B4, "B-4: ReduceLROnPlateau",       "B_exp4")


# Evaluación Final en Test Set

In [ ]:
# ── Evaluación final sobre el conjunto de test ──────────────────────────────
# Se usa model_B3 (mejor modelo global: B-3 Fine-tuning top 30, val_accuracy=0.8526)

print("=" * 55)
print("EVALUACIÓN FINAL — Test Set (10.000 imágenes)")
print("=" * 55)

test_loss_B3, test_acc_B3 = model_B3.evaluate(X_test, y_test, verbose=0)
print(f"Modelo:        B-3 (MobileNetV2 fine-tuning top 30)")
print(f"Test loss:     {test_loss_B3:.4f}")
print(f"Test accuracy: {test_acc_B3:.4f}")
print()

# Evaluación del mejor modelo A por completitud comparativa
test_loss_A4, test_acc_A4 = model_A4.evaluate(X_test, y_test, verbose=0)
print(f"Modelo:        A-4 (CNN + ReduceLROnPlateau)")
print(f"Test loss:     {test_loss_A4:.4f}")
print(f"Test accuracy: {test_acc_A4:.4f}")
print()

print("-" * 55)
print(f"Diferencia A→B en test: +{(test_acc_B3 - test_acc_A4)*100:.2f} pp")
print("=" * 55)


#Extraer Predicciones y Errores

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Predicciones sobre validacion
y_pred_probs = model_B3.predict(X_val, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_val.flatten()

# Indices donde el modelo falla
errores_idx = np.where(y_pred != y_true)[0]
print(f"Errores totales: {len(errores_idx)} de {len(y_true)} ({len(errores_idx)/len(y_true)*100:.1f}%)")

# Confianza del modelo en cada prediccion erronea
confianza_errores = y_pred_probs[errores_idx].max(axis=1)

# Ordenar por mayor confianza erronea (los peores fallos)
orden = np.argsort(confianza_errores)[::-1]
peores_idx = errores_idx[orden[:9]]

# Visualizar los 9 peores errores
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Peores errores del modelo B-3 (mayor confianza erronea)", fontsize=13)

for i, idx in enumerate(peores_idx):
    ax = axes[i // 3][i % 3]
    ax.imshow(X_val[idx])
    conf = confianza_errores[orden[i]] * 100
    ax.set_title(
        f"Real: {CLASES[y_true[idx]]}\n"
        f"Pred: {CLASES[y_pred[idx]]} ({conf:.1f}%)",
        fontsize=9,
        color='red'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig("error_analysis.png", dpi=120)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ── Tasa de error por clase ───────────────────────────────────────────────
print("Tasa de error por clase (model B-3, conjunto validación):")
print("-" * 45)
errores_por_clase = {}
for i, nombre in enumerate(CLASES):
    mask      = y_true == i
    n_total   = mask.sum()
    n_error   = (y_pred[mask] != i).sum()
    tasa      = n_error / n_total * 100
    errores_por_clase[nombre] = tasa
    print(f"  {nombre:12s}: {n_error:4d}/{n_total}  ({tasa:.1f}% error)")

# ── Pares más confundidos ─────────────────────────────────────────────────
print()
print("Top-5 pares más confundidos (real → predicho):")
print("-" * 45)
cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)          # ignorar aciertos
pares = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            pares.append((cm[i, j], CLASES[i], CLASES[j]))
pares.sort(reverse=True)
for count, real, pred in pares[:5]:
    print(f"  {real:12s} → {pred:12s}: {count} errores")


In [ ]:
# ── Mapa de calor de la matriz de confusión ───────────────────────
cm_full = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_full,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASES, yticklabels=CLASES,
    ax=ax
)
ax.set_title("Matriz de Confusión — Modelo B-3 (validación)", fontsize=13)
ax.set_ylabel("Clase real")
ax.set_xlabel("Clase predicha")
plt.tight_layout()
plt.savefig("checkpoints/confusion_matrix_B3.png", dpi=100)
plt.show()


## Patrón Identificado

El análisis de los 9 peores errores (mayor confianza errónea) junto con la matriz de confusión revela un **patrón sistemático de confusión entre clases visualmente similares**:

- **cat ↔ dog**: son el par más confundido. Ambas clases comparten textura de pelo, forma general y contexto de fondo (interiores domésticos). A 32×32 px, las diferencias morfológicas (hocico, orejas) quedan reducidas a pocos píxeles.
- **automobile ↔ truck**: comparten estructura de vehículo, perspectiva frontal y colores similares. La distinción clave (tamaño relativo, número de ejes) es difícil de capturar a baja resolución.
- **deer ↔ horse**: cuadrúpedos con proporciones similares y fondos naturales parecidos (pradera, campo).

El denominador común es que **los errores de alta confianza ocurren en clases semánticamente próximas**, no entre clases arbitrarias. El modelo ha aprendido representaciones parcialmente correctas pero no discrimina los rasgos finos que diferencian pares similares.


## Acciones Propuestas

**Acción 1 — Fine-tuning más profundo (descongelar top 60 capas) con LR bajo**

Los errores de alta confianza en pares similares (cat/dog, auto/truck) indican que las representaciones del backbone no están suficientemente especializadas en los rasgos discriminativos de CIFAR-10. Con solo 30 capas descongeladas (B-3), las capas medias del backbone siguen siendo genéricas de ImageNet. Descongelar las 60 últimas capas con un LR más bajo (1e-5) permitiría adaptar representaciones de nivel medio (texturas, bordes específicos) al dominio, potencialmente reduciendo la confusión en pares similares.

**Acción 2 — Label smoothing (ε = 0.1) para reducir sobreconfianza**

Los 9 peores errores son predicciones incorrectas realizadas con confianza muy alta (>90%). Esto indica que el modelo está mal calibrado: asigna probabilidades extremas a clases incorrectas en ejemplos ambiguos. Aplicar label smoothing (reemplazar targets one-hot por distribuciones suavizadas con ε=0.1) penaliza las distribuciones de probabilidad muy concentradas durante el entrenamiento, reduciendo la sobreconfianza sin comprometer la accuracy en ejemplos claros. Es un cambio de una línea en la función de pérdida y directamente motivado por este análisis de errores.
